# Field Scan : comptage des bananiers v2 (Kaggle)

Deux types de modèles comparés **sur les mêmes données et avec la même notation** :
- **DeepForest** : détecteur de boîtes autour des couronnes (pré-entraîné sur des couronnes d'arbres) ;
- **détecteur de centres** : un réseau qui produit une carte de probabilité, avec un pic au centre de chaque bananier (ResNet-18 pré-entraîné sur ImageNet + décodeur).

Données préparées par `outils/preparer_comptage.py` (toutes à 5 cm par pixel, un point par bananier) :

| Jeu | Apprentissage | Validation | Test (une seule fois, à la fin) |
|---|---|---|---|
| ReforesTree | Flora, Leonor, Guevara, Arteaga | Manuel | Nestor |
| Count banana plants | vols 0221, 0222 | vols 0223, 0224 | — |
| Plants-bana `segmento` | zone gauche | zone bas-droite ouest | zone bas-droite est + haut |
| Plants-bana `corte` | tout | — | — |

**Notation** (règle du sujet) : un bananier détecté est juste si son centre est à moins de 1 m (20 px) d'un vrai bananier, chaque vrai bananier ne pouvant être trouvé qu'une fois. F1 par jeu de validation, et erreur de comptage en %.

**Règle de choix, fixée avant les essais :** la **moyenne des F1 sur les 3 validations** (Manuel, Count banana, `segmento`). Pour chaque modèle, elle choisit l'époque et le seuil (et, pour DeepForest, le seuil de fusion des boîtes) ; puis elle départage les deux modèles.

**Avant de lancer :** GPU T4, Internet **On**, le dataset `comptage` (zip de `data/traite/comptage`) en Input. Puis **Save & Run All (Commit)**. Laisser `LANCER_TEST = False`.

In [ ]:
# 1. GPU et emplacement des données
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
from pathlib import Path

def chercher(nom):
    return sorted(Path(d) / nom for d, _, f in os.walk('/kaggle/input', followlinks=True) if nom in f)

RACINE = [p.parent for p in chercher('images.csv') if (p.parent / 'points.csv').exists()][0]
print('Données :', RACINE, sorted(p.name for p in RACINE.iterdir()))

In [ ]:
# 2. Réglages
import random, json, time, math, traceback
import numpy as np, pandas as pd, torch

MODELES = ['centres', 'deepforest']      # les deux essais de cette nuit, dans cet ordre
PIXEL_M = 0.05
RAYON_PX = 1.0 / PIXEL_M                 # 1 m = 20 pixels
SEUILS = np.round(np.arange(0.05, 0.96, 0.05), 2)

# Détecteur de centres
EPOQUES_CENTRES = 30
CROPS_PAR_EPOQUE = 2000                  # morceaux tirés au hasard à chaque époque
TAILLE_CROP = 448
LOT = 16
POIDS_JEUX = {'reforestree': 0.30, 'segmento': 0.30, 'count_banana': 0.25, 'corte': 0.15}   # part de chaque jeu dans un tour
FENETRE_PIC = 7                          # un pic = maximum dans 7 x 7 cellules de 20 cm (1,4 m)

# DeepForest
EPOQUES_DEEPFOREST = 10
TUILE_DF = 400
NMS_DF = [0.2, 0.3, 0.4, 0.5]            # recouvrement au-delà duquel deux boîtes sont fusionnées (testé sur la validation)

LANCER_TEST = False                      # True seulement pour le test final (Nestor + zone de test de segmento)
GRAINE = 0
random.seed(GRAINE); np.random.seed(GRAINE); torch.manual_seed(GRAINE)
appareil = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Calcul sur :', appareil, torch.cuda.get_device_name(0) if appareil == 'cuda' else '')
SORTIE = Path('/kaggle/working/comptage_v2'); SORTIE.mkdir(parents=True, exist_ok=True)

In [ ]:
# 3. Chargement des images et des bananiers
from PIL import Image

images = pd.read_csv(RACINE / 'images.csv')
points = pd.read_csv(RACINE / 'points.csv')
PIX, PTS = {}, {}
for n in images.image:
    PIX[n] = np.asarray(Image.open(RACINE / 'images' / n).convert('RGB'))
    p = points[points.image == n]
    PTS[n] = (p[['x', 'y']].to_numpy(np.float32), p.diametre.to_numpy(np.float32))

bilan = images.assign(bananiers=images.image.map(lambda n: len(PTS[n][0])))
print(bilan.groupby(['jeu', 'partie']).agg(images=('image', 'size'), bananiers=('bananiers', 'sum')).to_string())

VALIDATIONS = {
    'Manuel (ferme mélangée)': list(images[(images.jeu == 'reforestree') & (images.partie == 'validation')].image),
    'Count banana (bananeraie)': list(images[(images.jeu == 'count_banana') & (images.partie == 'validation')].image),
    'segmento (bananeraie dense)': list(images[(images.jeu == 'segmento') & (images.partie == 'validation')].image),
}
TESTS = {
    'Nestor (ferme mélangée)': list(images[(images.jeu == 'reforestree') & (images.partie == 'test')].image),
    'segmento (bananeraie dense)': list(images[(images.jeu == 'segmento') & (images.partie == 'test')].image),
}
APPRENTISSAGE = images[images.partie == 'apprentissage']
for nom, liste in VALIDATIONS.items():
    print(f"validation {nom:30s} {len(liste):3d} images, {sum(len(PTS[n][0]) for n in liste)} bananiers")

In [ ]:
# 4. Notation : centre à moins de 1 m d'un vrai bananier, chaque vrai bananier trouvé au plus une fois
from scipy.optimize import linear_sum_assignment

def apparier(det, vrai):
    if len(det) == 0 or len(vrai) == 0:
        return 0
    d = np.linalg.norm(det[:, None] - vrai[None], axis=2)
    cout = np.where(d <= RAYON_PX, d, 1e6)
    l, c = linear_sum_assignment(cout)
    return int((cout[l, c] <= RAYON_PX).sum())

def hors_image(n, xy):
    # Détections sur du noir (hors parcelle ReforesTree, bords ajoutés) : ignorées
    if len(xy) == 0:
        return np.zeros(0, bool)
    im = PIX[n]; x = np.clip(xy[:, 0].astype(int), 0, im.shape[1] - 1); y = np.clip(xy[:, 1].astype(int), 0, im.shape[0] - 1)
    return im[y, x].sum(1) == 0

def noter(detections, jeux):
    # detections : {image: (xy (n, 2), score (n,))} ; renvoie un tableau seuil x jeu, et la moyenne des F1
    lignes = []
    for nom, liste in jeux.items():
        for s in SEUILS:
            vp = nd = nv = 0
            for n in liste:
                xy, sc = detections[n]
                xy = xy[sc >= s]; xy = xy[~hors_image(n, xy)]
                vrai = PTS[n][0]
                vp += apparier(xy, vrai); nd += len(xy); nv += len(vrai)
            lignes.append({'jeu': nom, 'seuil': s, 'detections': nd, 'vrais': nv, 'vp': vp})
    t = pd.DataFrame(lignes)
    t['precision'] = t.vp / t.detections.clip(lower=1); t['rappel'] = t.vp / t.vrais.clip(lower=1)
    t['f1'] = 2 * t.vp / (t.detections + t.vrais).clip(lower=1)
    t['erreur_comptage'] = (t.detections - t.vrais) / t.vrais.clip(lower=1)
    moyenne = t.groupby('seuil').f1.mean()
    return t, moyenne

def resume(t, seuil):
    r = t[t.seuil == seuil].set_index('jeu')[['f1', 'precision', 'rappel', 'erreur_comptage', 'detections', 'vrais']]
    return r.round(3)

RESULTATS = {}   # un bilan par modèle

In [ ]:
# 5. Détecteur de centres
#    Réseau : ResNet-18 (ImageNet) + décodeur qui additionne ses 4 étages, du plus grossier au plus fin ;
#    sortie : une carte de probabilité, une cellule = 4 x 4 pixels (20 cm).
#    Cible : une petite bosse (gaussienne) centrée sur chaque bananier, de largeur proportionnelle à sa couronne.
#    Perte : perte focale de CenterNet (les cellules sans bananier sont très majoritaires).
import torch.nn as nn, torch.nn.functional as F
from torchvision import models
from torch.utils.data import Dataset, DataLoader

class DetecteurCentres(nn.Module):
    def __init__(self):
        super().__init__()
        r = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.souche = nn.Sequential(r.conv1, r.bn1, r.relu, r.maxpool)
        self.etages = nn.ModuleList([r.layer1, r.layer2, r.layer3, r.layer4])
        self.lateral = nn.ModuleList([nn.Conv2d(c, 64, 1) for c in (64, 128, 256, 512)])
        self.tete = nn.Sequential(nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(), nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
                                  nn.Conv2d(64, 1, 1))
        self.tete[-1].bias.data.fill_(-2.19)       # départ : probabilité 0,1 partout (usage CenterNet)
    def forward(self, x):
        x = self.souche(x); sorties = []
        for e in self.etages:
            x = e(x); sorties.append(x)
        y = self.lateral[3](sorties[3])
        for k in (2, 1, 0):
            y = F.interpolate(y, size=sorties[k].shape[-2:], mode='nearest') + self.lateral[k](sorties[k])
        return self.tete(y)                         # logits, 1/4 de la résolution

MOY = np.array([0.485, 0.456, 0.406], np.float32); ECT = np.array([0.229, 0.224, 0.225], np.float32)

def carte_cible(xy, diam, h, w):
    H, W = h // 4, w // 4
    carte = np.zeros((H, W), np.float32)
    yy, xx = np.mgrid[0:H, 0:W]
    for (x, y), d in zip(xy, diam):
        cx, cy = x / 4, y / 4
        if not (0 <= cx < W and 0 <= cy < H):
            continue
        s = max(d / 4 / 6, 0.8)                     # écart-type : 1/6 du diamètre, en cellules
        x0, x1 = int(max(0, cx - 3 * s)), int(min(W, cx + 3 * s + 1)); y0, y1 = int(max(0, cy - 3 * s)), int(min(H, cy + 3 * s + 1))
        g = np.exp(-((xx[y0:y1, x0:x1] - int(cx)) ** 2 + (yy[y0:y1, x0:x1] - int(cy)) ** 2) / (2 * s * s))
        carte[y0:y1, x0:x1] = np.maximum(carte[y0:y1, x0:x1], g)
    return carte

class Morceaux(Dataset):
    # Chaque élément : un morceau tiré au hasard (jeu tiré selon POIDS_JEUX, image tirée selon sa surface)
    def __init__(self, n):
        self.n = n
        self.par_jeu = {j: g.image.tolist() for j, g in APPRENTISSAGE.groupby('jeu') if j in POIDS_JEUX}
        self.poids_img = {j: np.array([PIX[i].shape[0] * PIX[i].shape[1] for i in l], float) for j, l in self.par_jeu.items()}
        self.jeux = list(self.par_jeu); p = np.array([POIDS_JEUX[j] for j in self.jeux]); self.p_jeux = p / p.sum()
    def __len__(self):
        return self.n
    def __getitem__(self, k):
        rng = np.random.default_rng()
        j = self.jeux[rng.choice(len(self.jeux), p=self.p_jeux)]
        w = self.poids_img[j]; n = self.par_jeu[j][rng.choice(len(w), p=w / w.sum())]
        im = PIX[n]; xy, diam = PTS[n]
        echelle = rng.uniform(0.8, 1.25)
        cote = int(round(TAILLE_CROP / echelle))
        h, wd = im.shape[:2]
        for _ in range(10):                          # évite les morceaux presque tout noirs (hors parcelle)
            x0 = rng.integers(0, max(1, wd - cote + 1)); y0 = rng.integers(0, max(1, h - cote + 1))
            if (im[y0:y0 + cote:8, x0:x0 + cote:8].max(2) == 0).mean() < 0.9:
                break
        morceau = np.zeros((cote, cote, 3), np.uint8)
        dec = im[y0:y0 + cote, x0:x0 + cote]; morceau[:dec.shape[0], :dec.shape[1]] = dec
        m = Image.fromarray(morceau).resize((TAILLE_CROP, TAILLE_CROP), Image.Resampling.BILINEAR)
        f = TAILLE_CROP / cote
        p = (xy - [x0, y0]) * f; d = diam * f
        a = np.asarray(m, np.float32) / 255
        # couleurs et lumière
        a = np.clip((a - 0.5) * rng.uniform(0.75, 1.25) + 0.5 + rng.uniform(-0.1, 0.1), 0, 1)
        gris = a.mean(2, keepdims=True); a = np.clip(gris + (a - gris) * rng.uniform(0.7, 1.3), 0, 1)
        # miroirs et quarts de tour
        if rng.random() < 0.5:
            a = a[:, ::-1]; p = np.c_[TAILLE_CROP - p[:, 0], p[:, 1]]
        for _ in range(rng.integers(0, 4)):
            a = np.rot90(a); p = np.c_[p[:, 1], TAILLE_CROP - p[:, 0]]
        cible = carte_cible(p, d, TAILLE_CROP, TAILLE_CROP)
        x = torch.from_numpy(((a - MOY) / ECT).transpose(2, 0, 1).copy())
        return x, torch.from_numpy(cible)[None]

def perte_focale(logits, cible):
    p = torch.sigmoid(logits).clamp(1e-4, 1 - 1e-4)
    pos = cible.eq(1).float(); neg = 1 - pos
    l_pos = -(torch.log(p) * (1 - p) ** 2 * pos).sum()
    l_neg = -(torch.log(1 - p) * p ** 2 * (1 - cible) ** 4 * neg).sum()
    return (l_pos + l_neg) / pos.sum().clamp(min=1)

@torch.no_grad()
def detecter_centres(modele, n):
    im = PIX[n]; h, w = im.shape[:2]
    H, W = math.ceil(h / 32) * 32, math.ceil(w / 32) * 32
    a = np.zeros((H, W, 3), np.float32); a[:h, :w] = im / 255
    x = torch.from_numpy(((a - MOY) / ECT).transpose(2, 0, 1).copy())[None].to(appareil)
    with torch.autocast(appareil, enabled=appareil == 'cuda'):
        c = torch.sigmoid(modele(x).float())
    pic = F.max_pool2d(c, FENETRE_PIC, stride=1, padding=FENETRE_PIC // 2)
    garde = (c == pic) & (c >= SEUILS.min())
    yy, xx = torch.nonzero(garde[0, 0], as_tuple=True)
    sc = c[0, 0, yy, xx].cpu().numpy(); xy = np.c_[(xx.cpu().numpy() + 0.5) * 4, (yy.cpu().numpy() + 0.5) * 4]
    dans = (xy[:, 0] < w) & (xy[:, 1] < h)
    return xy[dans].astype(np.float32), sc[dans]

def entrainer_centres():
    modele = DetecteurCentres().to(appareil)
    souche = list(modele.souche.parameters()) + list(modele.etages.parameters())
    neuf = list(modele.lateral.parameters()) + list(modele.tete.parameters())
    opt = torch.optim.AdamW([{'params': souche, 'lr': 3e-4}, {'params': neuf, 'lr': 1e-3}], weight_decay=1e-4)
    charge = DataLoader(Morceaux(CROPS_PAR_EPOQUE), LOT, num_workers=4, persistent_workers=True)
    plan = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[3e-4, 1e-3], total_steps=EPOQUES_CENTRES * len(charge))
    echelle = torch.amp.GradScaler(enabled=appareil == 'cuda')
    meilleur, historique = (-1, None, None), []
    for e in range(1, EPOQUES_CENTRES + 1):
        modele.train(); debut = time.time(); total = 0
        for x, y in charge:
            x, y = x.to(appareil), y.to(appareil)
            with torch.autocast(appareil, enabled=appareil == 'cuda'):
                logits = modele(x)
            l = perte_focale(logits.float(), y)
            opt.zero_grad(); echelle.scale(l).backward(); echelle.unscale_(opt)
            nn.utils.clip_grad_norm_(modele.parameters(), 5.0); echelle.step(opt); echelle.update(); plan.step()
            total += l.item()
        assert math.isfinite(total), 'perte NaN'
        modele.eval()
        det = {n: detecter_centres(modele, n) for l_ in VALIDATIONS.values() for n in l_}
        t, moy = noter(det, VALIDATIONS)
        s = float(moy.idxmax()); f1s = t[t.seuil == s].set_index('jeu').f1
        historique.append({'epoque': e, 'perte': total / len(charge), 'seuil': s, 'f1_moyenne': float(moy.max()),
                           **{k: round(float(v), 3) for k, v in f1s.items()}})
        print(f"époque {e:2d}  perte {total / len(charge):.3f}  F1 moyenne {moy.max():.3f} (seuil {s})  "
              + '  '.join(f'{k.split()[0]} {v:.3f}' for k, v in f1s.items()) + f'  ({time.time() - debut:.0f} s)')
        if moy.max() > meilleur[0]:
            meilleur = (float(moy.max()), e, s)
            torch.save(modele.state_dict(), SORTIE / 'centres.pt')
    modele.load_state_dict(torch.load(SORTIE / 'centres.pt'))
    modele.eval()
    return modele, meilleur, pd.DataFrame(historique)

if 'centres' in MODELES:
    try:
        debut = time.time()
        modele_centres, (f1_c, epoque_c, seuil_c), hist_c = entrainer_centres()
        det_c = {n: detecter_centres(modele_centres, n) for l_ in VALIDATIONS.values() for n in l_}
        t_c, moy_c = noter(det_c, VALIDATIONS)
        RESULTATS['centres'] = {'f1_moyenne': f1_c, 'epoque': epoque_c, 'seuil': seuil_c,
                                'par_jeu': resume(t_c, seuil_c).reset_index().to_dict('records'),
                                'minutes': round((time.time() - debut) / 60, 1)}
        hist_c.to_csv(SORTIE / 'centres_historique.csv', index=False); t_c.to_csv(SORTIE / 'centres_seuils.csv', index=False)
        print(f'\nDétecteur de centres retenu : époque {epoque_c}, seuil {seuil_c}, F1 moyenne {f1_c:.3f}')
        print(resume(t_c, seuil_c).to_string())
    except Exception:
        traceback.print_exc(); RESULTATS['centres'] = {'erreur': traceback.format_exc()[-1500:]}
    json.dump(RESULTATS, open(SORTIE / 'resultats.json', 'w'), indent=2, ensure_ascii=False, default=float)

In [ ]:
# 6. DeepForest : tuiles de 400 px pour l'apprentissage (boîte = centre +/- diamètre / 2),
#    prédiction par fenêtres de 400 px qui se recouvrent, puis fusion des boîtes en double sur l'image entière.
#    Comme pour le détecteur de centres, on note la validation à la fin de chaque époque et on garde la meilleure.
def tuile(im, y0, x0):
    # Morceau de 400 x 400 ; les images plus petites (Count banana, corte : environ 240 px) sont complétées par du noir,
    # pour que le réseau voie toujours les bananiers à la même taille (il agrandit chaque entrée à 800 px)
    t = np.zeros((TUILE_DF, TUILE_DF, 3), np.uint8); m = im[y0:y0 + TUILE_DF, x0:x0 + TUILE_DF]
    t[:m.shape[0], :m.shape[1]] = m
    return t

def preparer_deepforest():
    dossier = Path('/kaggle/working/df_tuiles'); dossier.mkdir(parents=True, exist_ok=True)
    lignes, par_jeu = [], {}
    for _, r in APPRENTISSAGE.iterrows():
        im = PIX[r.image]; xy, diam = PTS[r.image]; h, w = im.shape[:2]
        ys = [0] if h <= TUILE_DF else sorted(set(list(range(0, h - TUILE_DF + 1, 300)) + [h - TUILE_DF]))
        xs = [0] if w <= TUILE_DF else sorted(set(list(range(0, w - TUILE_DF + 1, 300)) + [w - TUILE_DF]))
        for y0 in ys:
            for x0 in xs:
                if (im[y0:y0 + TUILE_DF, x0:x0 + TUILE_DF].max(2) == 0).mean() > 0.9:
                    continue
                t = tuile(im, y0, x0)
                nom = f'{Path(r.image).stem}_{y0}_{x0}.png'; Image.fromarray(t).save(dossier / nom)
                dedans = (xy[:, 0] >= x0) & (xy[:, 0] < x0 + TUILE_DF) & (xy[:, 1] >= y0) & (xy[:, 1] < y0 + TUILE_DF)
                for (x, y), d in zip(xy[dedans], diam[dedans]):
                    lignes.append({'image_path': nom, 'xmin': max(0, round(x - x0 - d / 2)), 'ymin': max(0, round(y - y0 - d / 2)),
                                   'xmax': min(min(w, x0 + TUILE_DF) - x0, round(x - x0 + d / 2)),
                                   'ymax': min(min(h, y0 + TUILE_DF) - y0, round(y - y0 + d / 2)),
                                   'label': 'Tree'})
                if not dedans.any():                 # tuile sans bananier : codée 0, 0, 0, 0 (convention DeepForest)
                    lignes.append({'image_path': nom, 'xmin': 0, 'ymin': 0, 'xmax': 0, 'ymax': 0, 'label': 'Tree'})
                par_jeu.setdefault(r.jeu, []).append(nom)
    df = pd.DataFrame(lignes)
    # Même part de chaque jeu que pour le détecteur de centres : on répète les tuiles des jeux peu représentés
    total = sum(len(v) for v in par_jeu.values()); morceaux = []
    for j, noms in par_jeu.items():
        rep = max(1, round(POIDS_JEUX[j] * total / len(noms)))
        for k in range(rep):
            d = df[df.image_path.isin(noms)].copy()
            if k:
                for n in noms:
                    (dossier / f'r{k}_{n}').write_bytes((dossier / n).read_bytes())
                d['image_path'] = f'r{k}_' + d.image_path
            morceaux.append(d)
        print(f'  {j:13s} {len(noms):5d} tuiles x {rep}')
    pd.concat(morceaux).to_csv('/kaggle/working/df_apprentissage.csv', index=False)
    return dossier

@torch.no_grad()
def boites_deepforest(reseau, n):
    # Toutes les boîtes de l'image (score >= plus petit seuil testé), fenêtre par fenêtre, sur le GPU
    en_apprentissage = reseau.training; reseau.eval()
    reseau.score_thresh = float(SEUILS.min()); reseau.nms_thresh = max(NMS_DF); reseau.detections_per_img = 500
    dev = next(reseau.parameters()).device
    im = PIX[n]; h, w = im.shape[:2]; pas = 300
    ys = [0] if h <= TUILE_DF else sorted(set(list(range(0, h - TUILE_DF + 1, pas)) + [h - TUILE_DF]))
    xs = [0] if w <= TUILE_DF else sorted(set(list(range(0, w - TUILE_DF + 1, pas)) + [w - TUILE_DF]))
    fenetres = [(y0, x0) for y0 in ys for x0 in xs if (im[y0:y0 + TUILE_DF, x0:x0 + TUILE_DF].max(2) == 0).mean() <= 0.95]
    boites, scores = [np.zeros((0, 4), np.float32)], [np.zeros(0, np.float32)]
    for k in range(0, len(fenetres), 8):
        lot = fenetres[k:k + 8]
        x = [torch.from_numpy(tuile(im, y0, x0)).permute(2, 0, 1).float().div(255).to(dev) for y0, x0 in lot]
        for (y0, x0), s in zip(lot, reseau(x)):
            boites.append(s['boxes'].float().cpu().numpy() + np.float32([x0, y0, x0, y0])); scores.append(s['scores'].float().cpu().numpy())
    reseau.train(en_apprentissage)
    return np.concatenate(boites), np.concatenate(scores)

def centres_apres_fusion(b, s, nms):
    import torchvision
    if len(b) == 0:
        return np.zeros((0, 2), np.float32), np.zeros(0, np.float32)
    garde = torchvision.ops.nms(torch.from_numpy(b), torch.from_numpy(s), nms).numpy()
    b, s = b[garde], s[garde]
    return np.c_[(b[:, 0] + b[:, 2]) / 2, (b[:, 1] + b[:, 3]) / 2].astype(np.float32), s

def detecter_deepforest(reseau, n, nms):
    return centres_apres_fusion(*boites_deepforest(reseau, n), nms)

def noter_deepforest(reseau):
    brut = {n: boites_deepforest(reseau, n) for l_ in VALIDATIONS.values() for n in l_}
    resultats = {}
    for nms in NMS_DF:
        t, moy = noter({n: centres_apres_fusion(b, s, nms) for n, (b, s) in brut.items()}, VALIDATIONS)
        resultats[nms] = (t, moy)
    return resultats

if 'deepforest' in MODELES:
    try:
        debut = time.time()
        import subprocess, sys
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'deepforest>=2,<3'], check=True)
        from deepforest import main as df_main
        from pytorch_lightning import Callback
        from pytorch_lightning.loggers import CSVLogger

        class SuiviEpoques(Callback):
            # À la fin de chaque époque : F1 sur les 3 validations, pour chaque seuil de fusion ; on garde le meilleur
            def __init__(self):
                self.meilleur, self.historique = (-1, None, None, None), []
            def on_train_epoch_end(self, trainer, module):
                e = trainer.current_epoch + 1; ligne = {'epoque': e}
                for nms, (t, moy) in noter_deepforest(module.model).items():
                    ligne[f'f1_moyenne_nms{nms}'] = round(float(moy.max()), 3)
                    if moy.max() > self.meilleur[0]:
                        self.meilleur = (float(moy.max()), e, nms, float(moy.idxmax()))
                        torch.save(module.model.state_dict(), SORTIE / 'deepforest.pt')
                self.historique.append(ligne)
                print(f'époque {e:2d}  ' + '  '.join(f'{k} {v}' for k, v in ligne.items() if k != 'epoque'), flush=True)

        dossier_df = preparer_deepforest()
        torch.manual_seed(GRAINE)
        modele_df = df_main.deepforest()
        modele_df.load_model('weecology/deepforest-tree')
        modele_df.config.train.csv_file = '/kaggle/working/df_apprentissage.csv'
        modele_df.config.train.root_dir = str(dossier_df)
        modele_df.config.train.epochs = EPOQUES_DEEPFOREST
        modele_df.config.train.augmentations = ['HorizontalFlip', 'VerticalFlip', 'RandomBrightnessContrast']
        modele_df.config.batch_size = 8; modele_df.config.workers = 2
        suivi = SuiviEpoques()
        journal = CSVLogger('/kaggle/working/journaux', name='deepforest')
        modele_df.create_trainer(logger=journal, callbacks=[suivi]); modele_df.trainer.fit(modele_df)
        j = pd.read_csv(Path(journal.log_dir) / 'metrics.csv')
        pertes = j.dropna(subset=['train_loss_epoch']) if 'train_loss_epoch' in j else j.iloc[0:0]
        print(pertes[['epoch', 'train_loss_epoch']].to_string(index=False))
        assert len(pertes) and np.isfinite(pertes.train_loss_epoch).all(), 'perte absente ou NaN'

        f1_d, epoque_d, nms_d, seuil_d = suivi.meilleur
        reseau_df = modele_df.model.to(appareil)
        reseau_df.load_state_dict(torch.load(SORTIE / 'deepforest.pt', map_location=appareil)); reseau_df.eval()
        t_d, moy_d = noter_deepforest(reseau_df)[nms_d]
        RESULTATS['deepforest'] = {'f1_moyenne': f1_d, 'epoque': epoque_d, 'nms': nms_d, 'seuil': seuil_d,
                                   'par_jeu': resume(t_d, seuil_d).reset_index().to_dict('records'),
                                   'minutes': round((time.time() - debut) / 60, 1)}
        pd.DataFrame(suivi.historique).to_csv(SORTIE / 'deepforest_historique.csv', index=False)
        t_d.to_csv(SORTIE / 'deepforest_seuils.csv', index=False)
        print(f'\nDeepForest retenu : époque {epoque_d}, fusion {nms_d}, seuil {seuil_d}, F1 moyenne {f1_d:.3f}')
        print(resume(t_d, seuil_d).to_string())
    except Exception:
        traceback.print_exc(); RESULTATS['deepforest'] = {'erreur': traceback.format_exc()[-1500:]}
    json.dump(RESULTATS, open(SORTIE / 'resultats.json', 'w'), indent=2, ensure_ascii=False, default=float)

In [ ]:
# 7. Comparaison sur la validation, et modèle retenu (règle : meilleure moyenne des 3 F1)
lignes = []
for m, r in RESULTATS.items():
    if 'erreur' in r:
        print(f'{m} : ÉCHEC\n{r["erreur"]}'); continue
    for d in r['par_jeu']:
        lignes.append({'modele': m, **d})
comparaison = pd.DataFrame(lignes)
if len(comparaison):
    print(comparaison.pivot(index='jeu', columns='modele', values='f1').round(3).to_string())
    print('\nErreur de comptage (détectés - vrais) / vrais :')
    print(comparaison.pivot(index='jeu', columns='modele', values='erreur_comptage').round(3).to_string())
    valides = {m: r for m, r in RESULTATS.items() if 'erreur' not in r}
    MODELE_RETENU = max(valides, key=lambda m: valides[m]['f1_moyenne'])
    print('\nF1 moyenne :', {m: round(r['f1_moyenne'], 3) for m, r in valides.items()})
    print('Modèle retenu :', MODELE_RETENU)
    RESULTATS['retenu'] = MODELE_RETENU
    comparaison.to_csv(SORTIE / 'comparaison_validation.csv', index=False)
json.dump(RESULTATS, open(SORTIE / 'resultats.json', 'w'), indent=2, ensure_ascii=False, default=float)

In [ ]:
# 8. Contrôle visuel : détections (jaune) et vrais bananiers (rouge) sur un morceau de chaque validation
import matplotlib.pyplot as plt
try:
    fig, axes = plt.subplots(len(VALIDATIONS), len(valides), figsize=(7 * len(valides), 7 * len(VALIDATIONS)), squeeze=False)
    for i, (nom, liste) in enumerate(VALIDATIONS.items()):
        n = max(liste, key=lambda k: len(PTS[k][0]))
        for j, m in enumerate(valides):
            if m == 'centres':
                xy, sc = detecter_centres(modele_centres, n); xy = xy[sc >= valides[m]['seuil']]
            else:
                xy, sc = detecter_deepforest(reseau_df, n, valides[m]['nms']); xy = xy[sc >= valides[m]['seuil']]
            im = PIX[n]; c = min(800, im.shape[0], im.shape[1]); y0 = (im.shape[0] - c) // 2; x0 = (im.shape[1] - c) // 2
            ax = axes[i, j]; ax.imshow(im[y0:y0 + c, x0:x0 + c]); ax.set_title(f'{m} - {nom}'); ax.axis('off')
            v = PTS[n][0]
            for pts, coul, taille in [(v, 'red', 40), (xy, 'yellow', 12)]:
                k = (pts[:, 0] >= x0) & (pts[:, 0] < x0 + c) & (pts[:, 1] >= y0) & (pts[:, 1] < y0 + c)
                ax.scatter(pts[k, 0] - x0, pts[k, 1] - y0, s=taille, c=coul, edgecolors='black', linewidths=0.5)
    plt.tight_layout(); plt.savefig(SORTIE / 'controle_visuel.png', dpi=80); plt.show()
except Exception:
    traceback.print_exc()

## Test final

À lancer **une seule fois**, quand le modèle est choisi : `LANCER_TEST = True`, puis ne plus rien changer.

In [ ]:
# 9. Test : Nestor (ferme mélangée) et zone de test de segmento (bananeraie dense)
if not LANCER_TEST:
    print('Test non lancé : mettre LANCER_TEST = True seulement pour le test final')
else:
    m = RESULTATS['retenu']; r = RESULTATS[m]
    if m == 'centres':
        det = {n: detecter_centres(modele_centres, n) for l_ in TESTS.values() for n in l_}
    else:
        det = {n: detecter_deepforest(reseau_df, n, r['nms']) for l_ in TESTS.values() for n in l_}
    t, _ = noter(det, TESTS)
    print(f'TEST, modèle {m}, seuil {r["seuil"]} (choisi sur la validation) :')
    print(resume(t, r['seuil']).to_string())
    RESULTATS['test'] = resume(t, r['seuil']).reset_index().to_dict('records')
    json.dump(RESULTATS, open(SORTIE / 'resultats.json', 'w'), indent=2, ensure_ascii=False, default=float)